<a href="https://colab.research.google.com/github/karthikkavali7/NLP/blob/main/NLP_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

df = pd.read_csv("Times_of_India_Healines_since_jan_2020_score.csv")

print(df.head())
print(df.shape)

   index  Unnamed: 0  S_No                                                URL  \
0      0           0   0.0  https://timesofindia.indiatimes.com/2020/1/1/a...   
1      1           1   1.0  https://timesofindia.indiatimes.com/2020/1/1/a...   
2      2           2   2.0  https://timesofindia.indiatimes.com/2020/1/1/a...   
3      3           3   3.0  https://timesofindia.indiatimes.com/2020/1/1/a...   
4      4           4   4.0  https://timesofindia.indiatimes.com/2020/1/1/a...   

                  Date                                           Headline  \
0  2020-01-01 00:00:00  Shivin Narang injures his hand on the set of h...   
1  2020-01-01 00:00:00             Allergy cases on the rise in Bengaluru   
2  2020-01-01 00:00:00      A grand Hanukkah celebration held in the city   
3  2020-01-01 00:00:00  I respect my competitors, because they bring o...   
4  2020-01-01 00:00:00  Strong New Year resolutions keep young minds m...   

                                       Headline Li

In [4]:
df.info()
df.isnull().sum()
df[['Positive','Negative','Neutral']].describe()
df['Headline'].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24924 entries, 0 to 24923
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   index          24924 non-null  int64  
 1   Unnamed: 0     24924 non-null  int64  
 2   S_No           24923 non-null  float64
 3   URL            24923 non-null  object 
 4   Date           24923 non-null  object 
 5   Headline       24923 non-null  object 
 6   Headline Link  15518 non-null  object 
 7   Positive       24923 non-null  float64
 8   Negative       24923 non-null  float64
 9   Neutral        24923 non-null  float64
 10  Compound       24923 non-null  float64
dtypes: float64(5), int64(2), object(4)
memory usage: 2.1+ MB


,Headline
0,Shivin Narang injures his hand on the set of h...
1,Allergy cases on the rise in Bengaluru
2,A grand Hanukkah celebration held in the city
3,"I respect my competitors, because they bring o..."
4,Strong New Year resolutions keep young minds m...


In [5]:
pip install nltk


In [6]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [7]:
def clean_text(text):

    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [9]:
df['clean_headline'] = df['Headline'].fillna('').apply(clean_text)

In [10]:
def get_sentiment(row):

    if row['Compound'] > 0.05:
        return "Positive"

    elif row['Compound'] < -0.05:
        return "Negative"

    else:
        return "Neutral"

df['sentiment'] = df.apply(get_sentiment, axis=1)

In [11]:
df['sentiment'].value_counts()

,count
sentiment,
Neutral,10930
Negative,8591
Positive,5403


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_headline'])
y = df['sentiment']

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

LogisticRegression()

In [17]:
y_pred = model.predict(X_test)

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred, average='weighted')

recall = recall_score(y_test, y_pred, average='weighted')

f1 = f1_score(y_test, y_pred, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.8363089267803411
Precision: 0.8486875061056833
Recall: 0.8363089267803411
F1 Score: 0.8336821923772275


In [20]:
def predict_news(news):

    news_clean = clean_text(news)

    vector = vectorizer.transform([news_clean])

    prediction = model.predict(vector)

    return prediction

print(predict_news("Government launches new startup policy"))

['Neutral']
